In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!nvidia-smi

Tue Jul 21 09:52:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]"

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27743, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 27743 (delta 47), reused 25 (delta 25), pack-reused 27637 (from 3)
Receiving objects: 100% (27743/27743), 13.61 MiB | 23.66 MiB/s, done.
Resolving deltas: 100% (19783/19783), done.
/kaggle/working/LLaMA-Factory
Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 26.0 MB

In [4]:
!llamafactory-cli version

----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.6.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------


In [5]:
from huggingface_hub import list_repo_files

files = list_repo_files("Qwen/Qwen2.5-1.5B-Instruct")
print(files[:10])

['.gitattributes', 'LICENSE', 'README.md', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [7]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=Fa

In [8]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(model.__class__.__name__)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM


In [9]:
prompt = "Explique en quelques lignes ce qu'est le machine learning."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(response)

Le Machine Learning est une branche de l'Intelligence Artificielle qui vise à enseigner aux systèmes informatiques comment faire des décisions sans être explicitement programmés pour cela. Il s'appuie sur la théorie des algorithmes et la statistique pour permettre aux ordinateurs d'apprendre automatiquement les règles et modèles complexes nécessaires pour résoudre des problèmes ou effectuer des tâches. En d'autres termes, il consiste à former un système capable d'améliorer ses performances sur un problème spécifique au fil du temps, grâce à l'analyse et à l'interprétation des données.


In [10]:
import json

train_data = [
    {
        "instruction": "Quelle est la capitale du Maroc ?",
        "input": "",
        "output": "La capitale du Maroc est Rabat."
    },
    {
        "instruction": "Qui est le roi du Maroc ?",
        "input": "",
        "output": "Le roi du Maroc est Mohammed VI."
    },
    {
        "instruction": "Traduis en français",
        "input": "Artificial Intelligence",
        "output": "Intelligence artificielle"
    }
]

with open("train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

print("Dataset créé.")

Dataset créé.


In [11]:
import json

dataset_info = {
    "tp_maroc": {
        "file_name": "train.json",
        "formatting": "alpaca",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output"
        }
    }
}

with open("dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

print("dataset_info.json créé.")

dataset_info.json créé.


In [12]:
!find examples -name "*.yaml"

examples/inference/qwen3_lora_sft.yaml
examples/inference/qwen3.yaml
examples/inference/qwen3_full_sft.yaml
examples/inference/qwen3vl.yaml
examples/ascend/qwen3_5moe_lora_sft_fsdp2.yaml
examples/ascend/qwen3vlmoe_full_sft_fsdp2.yaml
examples/ascend/qwen3vlmoe_lora_sft_fsdp.yaml
examples/ascend/qwen3_5_full_sft_fsdp2.yaml
examples/ascend/qwen3_full_sft_fsdp2.yaml
examples/ascend/qwen3moe_full_sft_fsdp.yaml
examples/ktransformers/accelerate/fsdp2_kt_int8_8gpu.yaml
examples/ktransformers/accelerate/fsdp2_kt_int8_1gpu.yaml
examples/ktransformers/accelerate/fsdp2_kt_int4.yaml
examples/ktransformers/accelerate/fsdp2_kt_int8.yaml
examples/ktransformers/accelerate/fsdp2_kt_bf16.yaml
examples/ktransformers/train_lora/deepseek_v2_lora_sft_kt.yaml
examples/ktransformers/train_lora/qwen3_5moe_lora_sft_kt.yaml
examples/ktransformers/train_lora/qwen3moe_lora_sft_kt.yaml
examples/ktransformers/train_lora/deepseek_v3_lora_sft_kt.yaml
examples/v1/train_batching_strategy/train_full_fsdp2_dynamic_batchi

In [13]:
with open("examples/train_lora/qwen3_lora_sft.yaml") as f:
    print(f.read())

### model
model_name_or_path: Qwen/Qwen3-4B-Instruct-2507
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_target: all

### dataset
dataset: identity,alpaca_en_demo
template: qwen3_nothink
cutoff_len: 2048
max_samples: 1000
preprocessing_num_workers: 16
dataloader_num_workers: 4

### output
output_dir: saves/qwen3-4b/lora/sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none  # choices: [none, wandb, tensorboard, swanlab, mlflow]

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
resume_from_checkpoint: null

### eval
# eval_dataset: alpaca_en_demo
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 500



In [14]:
yaml_content = """
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

stage: sft
do_train: true

finetuning_type: lora
lora_rank: 8
lora_target: all

dataset: tp_maroc
dataset_dir: .

template: qwen

cutoff_len: 1024
max_samples: 5

output_dir: saves/qwen25-demo

logging_steps: 1
save_steps: 10

overwrite_output_dir: true
plot_loss: true

per_device_train_batch_size: 1
gradient_accumulation_steps: 4

learning_rate: 1e-4

num_train_epochs: 5

bf16: true

report_to: none
"""

with open("train.yaml", "w") as f:
    f.write(yaml_content)

print("train.yaml créé.")

train.yaml créé.


In [15]:
!llamafactory-cli train ./train.yaml


[INFO|2026-07-21 09:53:53] llamafactory.launcher:144 >> Initializing 2 distributed tasks at: 127.0.0.1:56457
W0721 09:53:54.929000 154 torch/distributed/run.py:852] 
W0721 09:53:54.929000 154 torch/distributed/run.py:852] *****************************************
W0721 09:53:54.929000 154 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0721 09:53:54.929000 154 torch/distributed/run.py:852] *****************************************
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/loc

In [16]:
!dir

assets		   docker    MANIFEST.in     saves     train.json
CITATION.cff	   docs      pyproject.toml  scripts   train.yaml
CLAUDE.md	   examples  README.md	     src
data		   LICENSE   README_zh.md    tests
dataset_info.json  Makefile  requirements    tests_v1


In [17]:
!dir saves

qwen25-demo
